# Staff.am IT Jobs Scraper

**Source:** [staff.am](https://staff.am) — Armenian IT job board  
**Category:** Software Development (category = 1)  
**Purpose:** Scrape all active IT job postings with full text for NLP skill extraction.

### Scraping strategy
1. Page through `staff.am/en/jobs?category=1&page=N` until empty
2. Parse `<script id="__NEXT_DATA__">` JSON embedded in each listing page
3. Build detail URLs from card slugs and category codes
4. Fetch each detail page and parse its `__NEXT_DATA__` JSON for rich content
5. Skip expired jobs (`deadline < today`); concatenate description + responsibilities + required_qualifications → `full_text`

### Page structure
- Listing page: Next.js SSR — full payload in `<script id="__NEXT_DATA__">` JSON
- Detail page: same pattern — `props.pageProps.job` object with all fields
- Fields: title, company, location, is_remote, employment_type, job_candidate_level, skills, deadline, salary, description, responsibilities, required_qualifications

### Ethics & robots.txt
- `robots.txt` only disallows `/API/*` (checked 2026-03-22); category listing and detail pages are allowed  
- Rate-limited: 1.5 s between requests  
- User-Agent identifies academic research purpose  

### Output files
- `data/raw/jobs/staffam_jobs_raw_v2.csv` — all extracted fields (raw)
- `data/processed/jobs/staffam_jobs_standardized.csv` — canonical NLP schema

In [1]:
import json
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date
from pathlib import Path

BASE_URL  = "https://staff.am"
CATEGORY  = 1          # Software Development
HEADERS   = {"User-Agent": "ThesisResearch/1.0 (Armenian IT curriculum alignment; academic use)"}
DELAY_S   = 1.5
TODAY     = date.today().isoformat()

BASE_DIR = Path.cwd()
RAW_DIR  = BASE_DIR / "data" / "raw" / "jobs"
PROC_DIR = BASE_DIR / "data" / "processed" / "jobs"

print(f"Today: {TODAY}")
print(f"Raw output:  {RAW_DIR}")
print(f"Proc output: {PROC_DIR}")

Today: 2026-03-22
Raw output:  /Users/lianaaghamalyan/thesis_data/data/raw/jobs
Proc output: /Users/lianaaghamalyan/thesis_data/data/processed/jobs


## Helper functions

In [2]:
def html_to_text(html_str):
    """Strip HTML tags and normalize whitespace."""
    if not html_str:
        return ""
    text = BeautifulSoup(str(html_str), "html.parser").get_text(" ", strip=True)
    return re.sub(r"\s+", " ", text).strip()


def get_en(field):
    """Extract English value from a multilingual dict, fallback to am/ru."""
    if isinstance(field, dict):
        return field.get("en") or field.get("am") or field.get("ru") or ""
    return str(field) if field else ""


def get_skills_list(skills_raw):
    """Extract skill title strings from the skills array."""
    if not skills_raw or not isinstance(skills_raw, list):
        return []
    return [get_en(s.get("title", "")).strip() for s in skills_raw if s.get("title")]


def card_to_detail_url(card):
    """Build detail page URL from card slug and category code."""
    slug_val = card.get("slug", {})
    slug = (slug_val.get("en") or slug_val.get("am") or slug_val.get("ru") or ""
            if isinstance(slug_val, dict) else str(slug_val).strip())
    cat = card.get("category", {})
    cat_code = cat.get("code", "jobs") if isinstance(cat, dict) else "jobs"
    return f"{BASE_URL}/en/jobs/{cat_code}/{slug}" if slug else None


def card_posting_date(card):
    """Extract ISO posting date from card activated_at field."""
    act = card.get("activated_at", {})
    ts = (act.get("staffam") or act.get("en") or ""
          if isinstance(act, dict) else str(act) if act else "")
    return ts[:10]


print("Helpers defined.")

Helpers defined.


## Step 1 — Discover jobs from listing pages

In [3]:
all_cards = []
page = 1
while True:
    url = f"{BASE_URL}/en/jobs?category={CATEGORY}&page={page}"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"  ERROR fetching listing page {page}: {e}")
        break

    soup = BeautifulSoup(resp.text, "html.parser")
    nd_tag = soup.find("script", id="__NEXT_DATA__")
    if not nd_tag:
        print(f"  No __NEXT_DATA__ on page {page} — stopping.")
        break

    data = json.loads(nd_tag.string)
    try:
        jobs_raw = data["props"]["pageProps"]["jobs"]
        jobs_on_page = jobs_raw if isinstance(jobs_raw, list) else jobs_raw.get("data", [])
    except (KeyError, TypeError):
        jobs_on_page = []

    if not jobs_on_page:
        print(f"  Page {page}: 0 jobs — done.")
        break

    print(f"  Page {page}: {len(jobs_on_page)} cards")
    all_cards.extend(jobs_on_page)
    page += 1
    time.sleep(DELAY_S)

# Build detail URL + posting_date pairs
detail_urls_with_dates = []
for card in all_cards:
    url = card_to_detail_url(card)
    posting_date = card_posting_date(card)
    if url:
        detail_urls_with_dates.append((url, posting_date))

print(f"\nTotal cards found: {len(all_cards)}")
print(f"Detail URLs to scrape: {len(detail_urls_with_dates)}")

  Page 1: 53 cards
  Page 2: 6 cards
  Page 3: 0 jobs — done.

Total cards found: 59
Detail URLs to scrape: 59


## Step 2 — Scrape detail pages

For each job page, parse `__NEXT_DATA__` JSON to extract:
- **title**, **company** (`companiesStruct[0].title`), **location** (`job_city.title`), **is_remote**, **employment_type**, **specialist_level** (`job_candidate_level.title`)
- **posting_date** (from listing card), **deadline**, **salary_from/to/currency**
- **skills_tags** — structured taxonomy tags from Staff.am (comma-separated)
- **description**, **responsibilities**, **required_qualifications** — HTML-stripped rich text
- **full_text** = description + responsibilities + required_qualifications (primary NLP field)

Jobs where `deadline < today` are skipped (expired).

In [4]:
records = []
expired = 0
errors  = 0

for i, (url, posting_date) in enumerate(detail_urls_with_dates, 1):
    print(f"  [{i}/{len(detail_urls_with_dates)}] {url}")
    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"    ERROR {url}: {e}")
        errors += 1
        time.sleep(DELAY_S)
        continue

    soup = BeautifulSoup(resp.text, "html.parser")
    nd_tag = soup.find("script", id="__NEXT_DATA__")
    if not nd_tag:
        print(f"    No __NEXT_DATA__: {url}")
        errors += 1
        time.sleep(DELAY_S)
        continue

    data = json.loads(nd_tag.string)
    try:
        job = data["props"]["pageProps"]["job"]
    except (KeyError, TypeError):
        print(f"    No job object: {url}")
        errors += 1
        time.sleep(DELAY_S)
        continue

    # Expiry check
    deadline = job.get("deadline", "") or ""
    if deadline and deadline < TODAY:
        print(f"    EXPIRED (deadline={deadline}): skipped")
        expired += 1
        time.sleep(DELAY_S)
        continue

    # Text fields
    desc      = html_to_text(get_en(job.get("description", "")))
    resp_text = html_to_text(get_en(job.get("responsibilities", "")))
    req_qual  = html_to_text(get_en(job.get("required_qualifications", "")))

    parts = [p for p in [desc, resp_text, req_qual] if p]
    full_text = "\n\n".join(parts)

    skills_tags = ", ".join(s for s in get_skills_list(job.get("skills")) if s)

    title      = html_to_text(get_en(job.get("title", "")))
    cand_level = get_en((job.get("job_candidate_level") or {}).get("title", ""))
    emp_type   = get_en((job.get("job_type") or {}).get("title", ""))

    company_raw = job.get("companiesStruct") or job.get("company") or {}
    if isinstance(company_raw, list) and company_raw:
        company_raw = company_raw[0]
    company_name = get_en(company_raw.get("title") or company_raw.get("name") or "") if isinstance(company_raw, dict) else ""

    city_raw = job.get("job_city") or {}
    location = get_en(city_raw.get("title", "")) if isinstance(city_raw, dict) else ""

    record = {
        "source":            "staff.am",
        "source_url":        url,
        "job_title":         title,
        "company_name":      company_name,
        "location":          location,
        "is_remote":         bool(job.get("is_remote")),
        "employment_type":   emp_type,
        "specialist_level":  cand_level,
        "deadline":          deadline,
        "salary_from":       job.get("salary_from") or "",
        "salary_to":         job.get("salary_to")   or "",
        "salary_currency":   job.get("salary_currency") or "",
        "description":       desc,
        "responsibilities":  resp_text,
        "required_qualifications": req_qual,
        "skills_tags":       skills_tags,
        "full_text":         full_text,
    }
    record["posting_date"] = posting_date
    records.append(record)
    time.sleep(DELAY_S)

print(f"\nRecords scraped: {len(records)}")
print(f"Expired (skipped): {expired}")
print(f"Errors: {errors}")

  [1/59] https://staff.am/en/jobs/software-development/front-end-developer-965
  [2/59] https://staff.am/en/jobs/software-development/senior-and-mid-aspnet-web-developer-10
  [3/59] https://staff.am/en/jobs/software-development/net-developer-569
  [4/59] https://staff.am/en/jobs/software-development/generative-ai-engineer-19
  [5/59] https://staff.am/en/jobs/software-development/react-native-developer-316
  [6/59] https://staff.am/en/jobs/software-development/junior-middle-front-end-and-back-end-developers-28
  [7/59] https://staff.am/en/jobs/software-development/lead-igaming-product-manager-42
  [8/59] https://staff.am/en/jobs/software-development/full-stack-tech-lead-igaming-43
  [9/59] https://staff.am/en/jobs/software-development/senior-power-bi-developer-88
  [10/59] https://staff.am/en/jobs/software-development/senior-net-c-software-engineer-91
  [11/59] https://staff.am/en/jobs/software-development/senior-net-developer-112
  [12/59] https://staff.am/en/jobs/software-development/

## Step 3 — Save outputs and review coverage

In [5]:
df = pd.DataFrame(records)

# Save raw
raw_path = RAW_DIR / "staffam_jobs_raw_v2.csv"
df.to_csv(raw_path, index=False, encoding="utf-8")
print(f"Raw saved: {raw_path} ({len(df)} rows)")

# Standardized
std_cols = [
    "source", "source_url", "job_title", "company_name", "location",
    "is_remote", "employment_type", "specialist_level",
    "posting_date", "deadline", "salary_from", "salary_to", "salary_currency",
    "skills_tags", "description", "responsibilities", "required_qualifications",
    "full_text",
]
std_df = df[[c for c in std_cols if c in df.columns]]
std_path = PROC_DIR / "staffam_jobs_standardized.csv"
std_df.to_csv(std_path, index=False, encoding="utf-8")
print(f"Standardized saved: {std_path} ({len(std_df)} rows)")

Raw saved: /Users/lianaaghamalyan/thesis_data/data/raw/jobs/staffam_jobs_raw_v2.csv (55 rows)
Standardized saved: /Users/lianaaghamalyan/thesis_data/data/processed/jobs/staffam_jobs_standardized.csv (55 rows)


In [6]:
print("=== FIELD COVERAGE ===")
check_cols = ["job_title", "company_name", "location", "employment_type", "specialist_level",
              "posting_date", "deadline", "skills_tags", "description",
              "responsibilities", "required_qualifications", "full_text"]
for col in check_cols:
    if col in std_df.columns:
        filled = std_df[col].replace("", pd.NA).notna().sum()
        pct = filled / len(std_df) * 100
        print(f"  {col:30s}: {filled:3d}/{len(std_df)} ({pct:.0f}%)")

print()
ft = std_df["full_text"].str.len()
print(f"full_text — Min: {ft.min()}  Median: {ft.median():.0f}  Max: {ft.max()}")
print()
print("=== SAMPLE JOB TITLES ===")
for _, r in std_df.head(10).iterrows():
print(f"  {r['job_title'][:48]:48s} | {r['specialist_level']:12s} | {r['company_name'][:25]}")

=== FIELD COVERAGE ===
  job_title                     :  55/55 (100%)
  company_name                  :  55/55 (100%)
  location                      :  55/55 (100%)
  employment_type               :  54/55 (98%)
  specialist_level              :  55/55 (100%)
  posting_date                  :  55/55 (100%)
  deadline                      :  55/55 (100%)
  skills_tags                   :  54/55 (98%)
  description                   :  55/55 (100%)
  responsibilities              :  52/55 (95%)
  required_qualifications       :  55/55 (100%)
  full_text                     :  55/55 (100%)

full_text — Min: 502  Median: 1755  Max: 3401

=== SAMPLE JOB TITLES ===
  Front end developer                              | Senior       | ACRA CREDIT REPORTING CJS
  Senior and Mid ASP.NET Web Developer             | Not defined  | EarnHCM LLC
  .Net Developer                                   | Mid level    | Acba Bank OJSC
  Generative AI Engineer                           | Senior       | Bosto